# Model training

### Import data and required packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [2]:
df = pd.read_csv('../../data/clean/clean_public_data.csv')

In [3]:
df.head()

,epglnren,sup_riscaldata,sup_raffrescata,vol_riscaldato,vol_raffrescato,sup_disperdente,rapsv,asolsut,presenza_clim_invernale,presenza_clim_estiva,...,anno_costruzione,zona_climatica,d_uso_Residenziale,tipologia_edilizia,tipologia_costruttiva,num_servizi,efficienza_media,potenza_tot,num_simulati,piano
0,153.92,53.41,0.00,179.54,0.00,34.83,0.1940,0.0125,True,False,...,before 1930,D,True,in linea,muratura portante,2,0.650000,0.00,2,upper
1,196.28,210.07,0.00,865.20,0.00,540.37,0.6246,0.0503,True,False,...,1946-1960,D,True,monofamiliare,muratura portante,2,0.835000,54.80,0,multi_floor
2,193.17,72.29,72.29,261.13,261.13,176.77,0.6769,0.0550,True,True,...,1992-2005,D,True,altro,c.a. con laterizi,3,0.600000,29.62,0,ground
3,173.49,47.90,0.00,202.85,0.00,108.34,0.5341,0.0412,True,False,...,before 1930,E,True,plurifamiliare,muratura portante,2,0.515000,1.50,1,ground
4,177.51,60.00,60.00,217.08,217.08,171.90,0.7918,0.0340,True,True,...,1961-1975,D,True,blocco,legno,3,0.716667,13.00,0,underground


### Define X and y then preprocess

In [4]:
X = df.drop(['epglnren'], axis=1)

In [5]:
y = df['epglnren']


In [6]:
# Column transformer
numerical_features = X.select_dtypes(include=['int64', 'float64', 'bool']).columns
categorical_features = X.select_dtypes(include=['object', 'category', 'str']).columns

print(f'features excluding target variable: {len(X.columns)}')
print(f'numerical features: {len(numerical_features)}')
print(f'categorical features: {len(categorical_features)}')

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", categorical_transformer, categorical_features),
        ("StandardScaler", numeric_transformer, numerical_features)
    ]
)

features excluding target variable: 21
numerical features: 16
categorical features: 5


In [7]:
X = preprocessor.fit_transform(X)

In [8]:
X.shape

(10869, 57)

In [9]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((8695, 57), (2174, 57))

### Create an evaluation function to present all metrics after model training

In [10]:
def evaluate_model(true, predicted):

    mse = mean_squared_error(true, predicted)
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2 = r2_score(true, predicted)
    
    return mae, mse, rmse, r2

In [11]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "KNN Regressor": KNeighborsRegressor(),
    "Decision Tree Regressor": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor(),
    "SVR": SVR(),
    "CatBoost Regressor": CatBoostRegressor(verbose=0),
    "XGBoost Regressor": XGBRegressor(verbose=0),
    "LightGBM Regressor": LGBMRegressor(verbose=0)
}
model_results = []
r2_scores = []

for i in range(len(list(models))):

    model = list(models.values())[i]
    model.fit(X_train, y_train) # train the model

    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # evaluate model performance
    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    model_results.append(list(models.keys())[i])

    print('\nModel performance on training set:')
    print(f'  MAE: {model_train_mae:.4f}')
    print(f'  MSE: {model_train_mse:.4f}')
    print(f'  RMSE: {model_train_rmse:.4f}')
    print(f'  R2: {model_train_r2:.4f}')
    print ('---' * 10)
    print('Model performance on test set:')
    print(f'  MAE: {model_test_mae:.4f}')
    print(f'  MSE: {model_test_mse:.4f}')
    print(f'  RMSE: {model_test_rmse:.4f}')
    print(f'  R2: {model_test_r2:.4f}')
    
    r2_scores.append(model_test_r2)

    print('=' * 20)
    print('\n')

Linear Regression

Model performance on training set:
  MAE: 601.4575
  MSE: 136204695.9042
  RMSE: 11670.6768
  R2: 0.0033
------------------------------
Model performance on test set:
  MAE: 483.3425
  MSE: 481845.4307
  RMSE: 694.1509
  R2: -29.7156


Ridge Regression

Model performance on training set:
  MAE: 601.1633
  MSE: 136204697.9681
  RMSE: 11670.6768
  R2: 0.0033
------------------------------
Model performance on test set:
  MAE: 482.9830
  MSE: 481068.6999
  RMSE: 693.5912
  R2: -29.6661


Lasso Regression

Model performance on training set:
  MAE: 594.1389
  MSE: 136206080.7582
  RMSE: 11670.7361
  R2: 0.0033
------------------------------
Model performance on test set:
  MAE: 474.6887
  MSE: 464338.3460
  RMSE: 681.4238
  R2: -28.5996


KNN Regressor

Model performance on training set:
  MAE: 307.6475
  MSE: 120242236.0165
  RMSE: 10965.5021
  R2: 0.1201
------------------------------
Model performance on test set:
  MAE: 68.4226
  MSE: 11063.6928
  RMSE: 105.1841
  R2:

c:\Users\AmirN\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:29:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\AmirN\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\AmirN\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [12]:
pd.DataFrame(list(zip(model_results, r2_scores)), columns=['Model', 'R2 Score']).sort_values(by='R2 Score', ascending=False)

,Model,R2 Score
3,KNN Regressor,0.294737
7,SVR,0.205402
6,AdaBoost Regressor,-3.172223
2,Lasso Regression,-28.599572
1,Ridge Regression,-29.666060
0,Linear Regression,-29.715573
10,LightGBM Regressor,-722.943694
8,CatBoost Regressor,-1643.952971
5,Random Forest Regressor,-4376.566248
9,XGBoost Regressor,-34749.139753
